In [0]:
%pip install pyyaml 

In [0]:
dbutils.library.restartPython()

In [0]:
import yaml

In [0]:
customers_df = spark.read.parquet("/Volumes/pei/bronze/customer/")
customers_df.printSchema()


In [0]:
display(orders_df)

In [0]:
orders_df = spark.read.parquet("/Volumes/pei/bronze/orders/")
orders_df.printSchema()

In [0]:
# orders_df = spark.read.parquet("/Volumes/pei/bronze/orders/")
# customers_df = spark.read.parquet("/Volumes/pei/bronze/customer/")
products_df = spark.read.parquet("/Volumes/pei/bronze/product/")

In [0]:
null_columns = [c for c in products_df.columns if products_df.filter(col(c).isNull()).count() > 0]
print(null_columns)

In [0]:
display(products_df)

In [0]:
# config_path = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/silver/products.yaml"

# with open(config_path, "r") as f:
#     config = yaml.safe_load(f)

In [0]:
def read_source(source_config):
    return (
        spark.read
        .format(source_config["format"])
        .load(source_config["path"])
    )

In [0]:
def apply_schema_and_rename(df, schema_config):
    for col_def in schema_config:
        source_col = col_def["source_name"]
        target_col = col_def["name"]
        dtype = col_def["type"]

        if source_col in df.columns:
            df = df.withColumnRenamed(source_col, target_col)
            df = df.withColumn(target_col, F.col(target_col).cast(dtype))
        else:
            raise Exception(f"Column {source_col} not found in source")

    return df

In [0]:
# def apply_transformations(df, transformations):
#     for t in transformations:
#         df = df.selectExpr("*", t["expr"])
#     return df

In [0]:
from pyspark.sql.functions import col, to_date

def apply_transformations(df, transformations):

    for t in transformations:
        t_type = t["type"]

        if t_type == "cast":
            for col_name, dtype in t["columns"].items():

                # If dtype is simple (string)
                if isinstance(dtype, str):
                    df = df.withColumn(col_name, col(col_name).cast(dtype))

                # If dtype is dict (has format)
                elif isinstance(dtype, dict):
                    if dtype["type"] == "date":
                        df = df.withColumn(
                            col_name,
                            to_date(col(col_name), dtype["format"])
                        )

        # other transformations unchanged...

    return df

In [0]:
def apply_deduplication(df, dedup_config):
    if not dedup_config:
        return df

    keys = dedup_config["keys"]
    order_col = dedup_config["order_by"]

    window = Window.partitionBy(*keys).orderBy(F.col(order_col).desc())
    

    df = (
        df.withColumn("rn", F.row_number().over(window))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

    return df

In [0]:
def select_columns(df, columns):
    return df.select(*columns)

In [0]:
def write_target(df, target_config):
    writer = (
        df.write
        .format(target_config["format"])
        .mode(target_config["mode"])
    )

    # Optional partitioning (not used for products)
    if "partition_by" in target_config:
        writer = writer.partitionBy(*target_config["partition_by"])

    # Write to path
    if "path" in target_config:
        writer.save(target_config["path"])

    # Register table
    if "table" in target_config:
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {target_config['table']}
            USING DELTA
            LOCATION '{target_config['path']}'
        """)

In [0]:
# def write_target(df, target_config):

#     table_name = f"{target_config['schema']}.{target_config['table']}"

#     # Ensure schema exists
#     spark.sql(f"CREATE DATABASE IF NOT EXISTS {target_config['schema']}")

#     writer = (
#         df.write
#         .format(target_config.get("format", "delta"))
#         .mode(target_config.get("mode", "overwrite"))
#     )

#     # Optional partitioning
#     if "partition_by" in target_config:
#         writer = writer.partitionBy(*target_config["partition_by"])

#     # Write + register table together
#     if "path" in target_config:
#         writer.option("path", target_config["path"]).saveAsTable(table_name)
#     else:
#         writer.saveAsTable(table_name)

In [0]:
def apply_quality_checks(df, rules):
    for rule in rules:
        col = rule["column"]
        condition = rule["rule"]

        if condition == "not_null":
            df = df.filter(F.col(col).isNotNull())
        else:
            df = df.filter(f"{col} {condition}")

    return df

In [0]:
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
def process_table(config):
    # config = load_yaml(config_path)
    print("started main")
    df = read_source(config["source"])
    df = apply_schema_and_rename(df, config["schema"])
    df = apply_transformations(df, config.get("transformations", []))
    df = apply_deduplication(df, config.get("deduplication"))
    df = select_columns(df, config["columns"]["select"])
    write_target(df, config["target"])
    # display(df)

In [0]:
import os

In [0]:
# Main fn
# config_path = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/silver/products.yaml"

# with open(config_path, "r") as f:
#     config = yaml.safe_load(f) 


CONFIG_DIR = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/silver/"

def load_all_configs(config_dir):
    configs = []

    for file in os.listdir(config_dir):
        if file.endswith(".yaml"):
            full_path = os.path.join(config_dir, file)
            with open(full_path, "r") as f:
                cfg = yaml.safe_load(f)

            cfg["_config_path"] = full_path
            configs.append(cfg)

    return configs

# process_table(config)

In [0]:
configs = load_all_configs(CONFIG_DIR)

for cfg in configs:
    try:
        # print(f"Processing: {cfg.get('target', {}).get('table')}")

        process_table(cfg)
        #print(f"process complete for ", {cfg.get("table_name")})
        # print(f"Success: {cfg.get('target', {}).get('table')}")

    except Exception as e:
        print(f"Failed: {cfg.get('target', {}).get('table')} → {str(e)}")

In [0]:
# %sql
# SHOW SCHEMAS IN pei

In [0]:
# %sql


# CREATE SCHEMA IF NOT EXISTS pei.silver;

In [0]:
# def ingest_dataset(config):

#     reader_fn = get_reader(config["format"])

#     df = reader_fn(spark, config)

#     df = standardize(df, config)

#     write_to_bronze(df, config)

In [0]:
# for dataset in config["datasets"]:
#     try:
#         ingest_dataset(dataset)
#     except Exception as e:
#         print(f"Failed: {dataset['name']} → {str(e)}")

In [0]:
# for table_name in config["table_name"]:
#     try:
#         process_table(table_name)
#     except Exception as e:
#         print(f"Failed: {table_name} → {str(e)}")

In [0]:
# %sql
# drop table if exists pei.silver.products

In [0]:
df = spark.read.format("delta").load("/Volumes/pei/silver/orders/")
df.count()